# DUNS ID Enrichment Notebook

**Purpose:** Load an Excel file from `data/input/excel_files/`, match it against the unified DUNS reference (`data/output/unified_duns_list.csv`) using **client number ↔ CS**, append DUNS IDs, and produce a full summary report.

| Section | What |
|---|---|
| 1 | Imports & paths |
| 2 | Load input Excel file |
| 3 | Load unified DUNS reference |
| 4 | Configure join keys |
| 5 | Match & enrich |
| 6 | **Summary — match rate, unmatched samples** |
| 7 | Export enriched file & review package |

## Section 1 — Imports & Paths

In [ ]:
import warnings
from pathlib import Path
from datetime import datetime

import pandas as pd

warnings.filterwarnings("ignore")

# ── Root paths ────────────────────────────────────────────────────────────────
NOTEBOOK_DIR     = Path().resolve()                           # duns_id/notebooks/
PROJECT_ROOT     = NOTEBOOK_DIR.parent                        # duns_id/

INPUT_EXCEL_DIR  = PROJECT_ROOT / "data" / "input" / "excel_files"
REFERENCE_DIR    = PROJECT_ROOT / "data" / "output"
OUTPUT_DIR       = PROJECT_ROOT / "data" / "output"
REVIEW_DIR       = PROJECT_ROOT / "review"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

# ── ✏️  UPDATE THESE TWO LINES ────────────────────────────────────────────────
EXCEL_FILENAME   = "your_input_file.xlsx"   # filename inside data/input/excel_files/
EXCEL_SHEET      = 0                        # sheet index (0 = first) or sheet name
# ─────────────────────────────────────────────────────────────────────────────

DUNS_REFERENCE_FILE = REFERENCE_DIR / "unified_duns_list.csv"
EXCEL_FILE          = INPUT_EXCEL_DIR / EXCEL_FILENAME
RUN_TS              = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

print(f"Input Excel   : {EXCEL_FILE}")
print(f"DUNS reference: {DUNS_REFERENCE_FILE}")
print(f"Output dir    : {OUTPUT_DIR}")

## Section 2 — Load Input Excel File

In [ ]:
if not EXCEL_FILE.exists():
    raise FileNotFoundError(
        f"\nExcel file not found: {EXCEL_FILE}"
        f"\n → Place your file in: {INPUT_EXCEL_DIR}"
        f"\n → Then update EXCEL_FILENAME in Section 1."
    )

excel_df = pd.read_excel(EXCEL_FILE, sheet_name=EXCEL_SHEET, dtype=str)

# Normalise: strip whitespace from all string columns
excel_df = excel_df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

print(f"Rows loaded : {len(excel_df):,}")
print(f"Columns ({len(excel_df.columns)}): {list(excel_df.columns)}")
excel_df.head(5)

## Section 3 — Load Unified DUNS Reference

In [ ]:
if not DUNS_REFERENCE_FILE.exists():
    raise FileNotFoundError(
        f"\nDUNS reference not found: {DUNS_REFERENCE_FILE}"
        "\n → Run scripts/consolidate_duns.py first to generate it."
    )

duns_ref = pd.read_csv(DUNS_REFERENCE_FILE, dtype=str)
duns_ref = duns_ref.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

print(f"DUNS reference rows : {len(duns_ref):,}")
print(f"Columns ({len(duns_ref.columns)})  : {list(duns_ref.columns)}")
duns_ref.head(5)

## Section 4 — Configure Join Keys

Set `EXCEL_CLIENT_COL` to the exact column name in your Excel file that holds the **client / customer number**.  
This will be matched against the `CS` column in the DUNS reference.

In [ ]:
# ── ✏️  UPDATE THIS ───────────────────────────────────────────────────────────
EXCEL_CLIENT_COL = "client_number"   # column in your Excel file to match on CS
# ─────────────────────────────────────────────────────────────────────────────

# Validate the column exists
if EXCEL_CLIENT_COL not in excel_df.columns:
    raise KeyError(
        f"\nColumn '{EXCEL_CLIENT_COL}' not found in Excel file."
        f"\nAvailable columns: {list(excel_df.columns)}"
        f"\nUpdate EXCEL_CLIENT_COL above."
    )

if "CS" not in duns_ref.columns:
    raise KeyError("'CS' column not found in DUNS reference. Check unified_duns_list.csv.")

print(f"Joining  Excel['{EXCEL_CLIENT_COL}']  →  DUNS ref['CS']")
print(f"Excel unique clients : {excel_df[EXCEL_CLIENT_COL].nunique():,}")
print(f"DUNS ref unique CS   : {duns_ref['CS'].nunique():,}")

## Section 5 — Match & Enrich

In [ ]:
# Columns to bring in from the DUNS reference
DUNS_COLS = ["CS", "duns", "duns_count", "BU", "IDS", "SDS"]
duns_ref_slim = duns_ref[[c for c in DUNS_COLS if c in duns_ref.columns]].copy()

# Left join: keep all Excel rows, append DUNS where found
enriched = excel_df.merge(
    duns_ref_slim.rename(columns={"CS": EXCEL_CLIENT_COL}),
    on=EXCEL_CLIENT_COL,
    how="left",
    suffixes=("", "_duns")
)

# Flag each row
enriched["duns_match_status"] = enriched["duns"].apply(
    lambda x: "MATCHED" if pd.notna(x) and str(x).strip() not in ("", "nan") else "UNMATCHED"
)

print(f"Total rows after join : {len(enriched):,}")
print(f"MATCHED               : {(enriched['duns_match_status'] == 'MATCHED').sum():,}")
print(f"UNMATCHED             : {(enriched['duns_match_status'] == 'UNMATCHED').sum():,}")
enriched.head(5)

## Section 6 — Match Summary

In [ ]:
# ── Config: which column holds the unique customer identifier ─────────────────
CUSTOMER_ID_COL = "CUSTOMER_IDENTIFIER"   # ✏️ update if your column name differs

# ── Row-level counts ──────────────────────────────────────────────────────────
total_rows     = len(enriched)
matched_rows   = (enriched["duns_match_status"] == "MATCHED").sum()
unmatched_rows = total_rows - matched_rows
pct_match_rows  = matched_rows / total_rows * 100 if total_rows else 0
pct_unmatch_rows = 100 - pct_match_rows

# Keep backward-compatible names used by export cell
total      = total_rows
matched    = matched_rows
unmatched  = unmatched_rows
pct_match  = pct_match_rows
pct_unmatch = pct_unmatch_rows

# ── Unique-customer counts ────────────────────────────────────────────────────
if CUSTOMER_ID_COL in enriched.columns:
    total_cust    = enriched[CUSTOMER_ID_COL].nunique()
    matched_cust  = enriched.loc[
        enriched["duns_match_status"] == "MATCHED", CUSTOMER_ID_COL
    ].nunique()
    unmatched_cust = total_cust - matched_cust
    pct_match_cust  = matched_cust / total_cust * 100 if total_cust else 0
    pct_unmatch_cust = 100 - pct_match_cust
    has_cust_col = True
else:
    has_cust_col = False
    print(f"⚠️  Column '{CUSTOMER_ID_COL}' not found — unique-customer summary skipped.")
    print(f"   Available columns: {list(enriched.columns)}")

# ── Row-level summary table ───────────────────────────────────────────────────
row_summary = pd.DataFrame({
    "Category"  : ["Total rows", "Matched rows (DUNS found)", "Unmatched rows (no DUNS)"],
    "Count"     : [total_rows, matched_rows, unmatched_rows],
    "Percentage": [100.0, round(pct_match_rows, 2), round(pct_unmatch_rows, 2)],
})

print("=" * 55)
print("         MATCH SUMMARY — ALL ROWS")
print("=" * 55)
print(row_summary.to_string(index=False))
print("=" * 55)

# ── Unique-customer summary table ─────────────────────────────────────────────
if has_cust_col:
    cust_summary = pd.DataFrame({
        "Category"  : [
            f"Total unique {CUSTOMER_ID_COL}",
            "Unique customers MATCHED",
            "Unique customers UNMATCHED",
        ],
        "Count"     : [total_cust, matched_cust, unmatched_cust],
        "Percentage": [100.0, round(pct_match_cust, 2), round(pct_unmatch_cust, 2)],
    })

    print(f"\n{'=' * 55}")
    print(f"  MATCH SUMMARY — UNIQUE {CUSTOMER_ID_COL}")
    print("=" * 55)
    print(cust_summary.to_string(index=False))
    print("=" * 55)

# ── Clients with multiple DUNS IDs ────────────────────────────────────────────
if "duns_count" in enriched.columns:
    multi_duns = enriched.loc[
        enriched["duns_match_status"] == "MATCHED", "duns_count"
    ].apply(lambda x: int(x) if pd.notna(x) else 0)
    multi_flag = (multi_duns > 1).sum()
    print(f"\nRows with multiple DUNS IDs : {multi_flag:,}")

# ── Sample of unmatched rows ──────────────────────────────────────────────────
unmatched_df = enriched[enriched["duns_match_status"] == "UNMATCHED"].reset_index(drop=True)

sample_cols = [EXCEL_CLIENT_COL]
if has_cust_col and CUSTOMER_ID_COL != EXCEL_CLIENT_COL:
    sample_cols.append(CUSTOMER_ID_COL)

print(f"\nSample of unmatched entries (up to 10 unique {CUSTOMER_ID_COL if has_cust_col else EXCEL_CLIENT_COL}):")
display(
    unmatched_df[sample_cols].drop_duplicates(
        subset=[CUSTOMER_ID_COL] if has_cust_col else [EXCEL_CLIENT_COL]
    ).head(10)
)

## Section 7 — Export Results

In [ ]:
OUTPUT_DIR = BASE_DIR / "data" / "output"
REVIEW_DIR = BASE_DIR / "review"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

input_stem = Path(EXCEL_FILENAME).stem

# ── 1. Full enriched file ─────────────────────────────────────────────────────
enriched_csv  = OUTPUT_DIR / f"enriched_{input_stem}_{RUN_TS}.csv"
enriched_xlsx = OUTPUT_DIR / f"enriched_{input_stem}_{RUN_TS}.xlsx"

enriched.to_csv(enriched_csv, index=False)
enriched.to_excel(enriched_xlsx, index=False)

print(f"✅ Enriched CSV  : {enriched_csv}")
print(f"✅ Enriched XLSX : {enriched_xlsx}")

# ── 2. Unmatched review file ──────────────────────────────────────────────────
review_xlsx = REVIEW_DIR / f"unmatched_{input_stem}_{RUN_TS}.xlsx"

unmatched_df.to_excel(review_xlsx, index=False)
print(f"✅ Unmatched XLSX: {review_xlsx}  ({len(unmatched_df):,} rows)")

# ── 3. Plain-text summary report ──────────────────────────────────────────────
report_path = OUTPUT_DIR / f"match_report_{input_stem}_{RUN_TS}.txt"

report_lines = [
    "=" * 55,
    "         MATCH SUMMARY REPORT",
    "=" * 55,
    f"Input file   : {EXCEL_FILENAME}",
    f"Run timestamp: {RUN_TS}",
    "",
    "── ROW-LEVEL ──────────────────────────────────────────",
    f"  Total rows       : {total_rows:,}",
    f"  Matched rows     : {matched_rows:,}  ({pct_match_rows:.2f}%)",
    f"  Unmatched rows   : {unmatched_rows:,}  ({pct_unmatch_rows:.2f}%)",
]

if has_cust_col:
    report_lines += [
        "",
        f"── UNIQUE {CUSTOMER_ID_COL} ──────────────────────────────",
        f"  Total unique customers   : {total_cust:,}",
        f"  Matched unique customers : {matched_cust:,}  ({pct_match_cust:.2f}%)",
        f"  Unmatched unique         : {unmatched_cust:,}  ({pct_unmatch_cust:.2f}%)",
    ]

report_lines.append("=" * 55)

report_path.write_text("\n".join(report_lines), encoding="utf-8")
print(f"✅ Report TXT   : {report_path}")

print("\nAll exports complete.")